# Ekstraksi Soal & Materi dari PDF/Ebook dengan VLM (Gemini)

Notebook ini membaca **setiap halaman PDF sebagai gambar** lalu memakai VLM
(Gemini Flash) untuk:

1. Memisahkan **`soal`** (problem) vs **`materi`** (teori/definisi/penjelasan).
2. Untuk soal: memisah **`question` / `answer` / `steps`** (cara).
3. Membuang noise: header/footer, nomor halaman, watermark, URL, nama penulis.
4. Mempertahankan notasi matematika sebagai **LaTeX** inline `$...$`.

Kenapa VLM (baca gambar) bukan `pdfplumber`? Ekstraksi teks merusak notasi math
(`(cid:18)`, pecahan kebalik) dan mencampur soal+jawaban. VLM yang melihat
halaman menangani ini jauh lebih baik.

**Alur pemakaian:** isi API key → jalankan sel prototype di beberapa PDF →
cek hasil → jalankan batch penuh.

> Backend di-abstraksi lewat `call_vlm()`. Mau pindah ke Qwen2.5-VL lokal/Kaggle?
> Cukup ganti satu fungsi; prompt & schema tetap sama.


## 1. Install dependencies

In [1]:
# Jalankan sekali. Aman diulang. Deps inti + filter pipeline (semua backend).
%pip install -q google-genai pymupdf pillow tqdm python-dotenv \
    sentence-transformers faiss-cpu langdetect


Note: you may need to restart the kernel to use updated packages.


### 1b. (Opsional) Dependencies VLM lokal / Kaggle

Lewati kalau pakai Gemini (API/CLI). Untuk **Qwen2.5-VL** (`USE_QWEN_VLM=True`):

**Cara pakai di Kaggle:**
1. Buat Notebook baru → upload file `.ipynb` ini (atau copy isinya).
2. Upload data: **+ Add Data → Upload** folder `raw` (atau zip-nya) sebagai Dataset → otomatis mount di `/kaggle/input/<nama>/`.
3. **Settings → Accelerator → GPU T4 ×2**, **Internet → ON**.
4. Di sel konfigurasi set `USE_QWEN_VLM = True`. Path RAW/OUT terisi otomatis (`/kaggle/input`, `/kaggle/working`).
5. Run sel install ini → konfig → backend → prototype.

Catatan: Qwen2.5-VL-7B 4-bit muat di T4 16GB. Di GPU lokal 8GB, ganti `MODEL_ID` ke `Qwen/Qwen2.5-VL-3B-Instruct`.

In [2]:
# Dependencies KHUSUS Qwen2.5-VL (GPU) — jalankan HANYA kalau USE_QWEN_VLM=True.
# Skip cell ini kalau pakai Gemini (API/CLI).
# Di Kaggle: Settings -> Accelerator = GPU T4 x2, Internet = ON. Torch/CUDA sudah ada.
%pip install -q -U "transformers>=4.49.0" accelerate bitsandbytes qwen-vl-utils

# ⚠️ Kalau habis install ini -> RESTART KERNEL (Run > Restart & Clear) sebelum lanjut.
# Tanpa restart, modul lama ke-cache -> error "cannot import name '_Ink' from 'PIL._typing'".
print(">>> (Qwen) SELESAI. Kalau dijalankan, RESTART KERNEL sebelum lanjut. <<<")


Note: you may need to restart the kernel to use updated packages.
>>> (Qwen) SELESAI. Kalau dijalankan, RESTART KERNEL sebelum lanjut. <<<


## 2. Konfigurasi

API key dibaca dari environment / file `.env` di root proyek
(`GEMINI_API_KEY=...`). **Jangan** hard-code key di notebook (`.env` sudah masuk
`.gitignore`).

Dapatkan key gratis di Google AI Studio.

**Sumber data (lokal vs Google Drive).** Default baca `data/raw/` lokal. Untuk
pakai data di Drive: install **Google Drive for Desktop**, "Add shortcut to My
Drive" folder `DATA_NLP`, lalu set `RAW_DIR_OVERRIDE` (opsional juga
`OUT_DIR_OVERRIDE`) ke path Drive yang ke-sync — atau set env var
`RAW_DIR` / `OUT_DIR` tanpa edit notebook. Folder Drive yang ke-sync
diperlakukan sebagai folder lokal biasa, jadi sisa notebook tidak berubah.

In [3]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()  # baca .env di cwd / root

ON_KAGGLE = os.path.exists("/kaggle")   # autodetect environment Kaggle

# ── Path ──────────────────────────────────────────────────────────────
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

# ── Sumber data: lokal / Google Drive / Kaggle ────────────────────────
# Lokal+Drive: set RAW_DIR_OVERRIDE atau env RAW_DIR ke path Drive (Drive Desktop).
# Kaggle: upload data sebagai Dataset -> otomatis ke-mount di /kaggle/input.
RAW_DIR_OVERRIDE = r""    # kosong = auto (Kaggle: /kaggle/input ; lokal: data/raw atau env RAW_DIR)
OUT_DIR_OVERRIDE = r"C:\Users\Henry\Documents\KULIAH\sem_4\NLP\FP\data\extracted_vlm"    # absolut lokal

_raw = os.environ.get("RAW_DIR") or RAW_DIR_OVERRIDE
_out = os.environ.get("OUT_DIR") or OUT_DIR_OVERRIDE

if _raw:
    RAW_DIR = Path(_raw).expanduser()
elif ON_KAGGLE:
    RAW_DIR = Path("/kaggle/input")            # list_pdfs pakai rglob -> nemu semua PDF di dataset
else:
    RAW_DIR = PROJECT_ROOT / "data" / "raw"

if _out:
    OUT_DIR = Path(_out).expanduser()
elif ON_KAGGLE:
    OUT_DIR = Path("/kaggle/working/extracted_vlm")
else:
    OUT_DIR = PROJECT_ROOT / "data" / "extracted_vlm"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ══ PILIH BACKEND (toggle) ════════════════════════════════════════════
# Prioritas: QWEN_VLM > GEMINI_CLI > Gemini API. Set salah satu True.
USE_QWEN_VLM   = False   # VLM lokal/Kaggle (Qwen2.5-VL via GPU). Unlimited tapi kualitas < Gemini.
USE_GEMINI_CLI = False   # Gemini CLI (auth OAuth). Butuh `gemini` login.
                         # Dua-duanya False = Gemini API key (butuh GEMINI_API_KEY di .env). <- AKTIF

if USE_QWEN_VLM:
    BACKEND       = "qwen"
    MODEL_ID      = "Qwen/Qwen2.5-VL-7B-Instruct"
    SLEEP_BETWEEN = 0.0
elif USE_GEMINI_CLI:
    BACKEND       = "gemini_cli"
    MODEL_ID      = "gemini-2.5-flash"
    SLEEP_BETWEEN = 1.0
else:
    BACKEND       = "gemini"
    MODEL_ID      = "gemini-3.1-flash-lite"  # GA stabil + MURAH ($0.10/$0.40). 503 jarang vs preview 3.1-flash-lite. Naik ke "gemini-2.5-flash" kalau masih 503.
                                          # Mau kualitas terbaik & kuota cukup: "gemini-2.5-flash" (5 RPM, naikkan SLEEP).
    SLEEP_BETWEEN = 0.0                    # paid + paralel -> tanpa sleep; retry handle 429/503

DPI         = 150        # naikkan ke 200 utk halaman padat (lebih akurat, lebih mahal/berat)
TEMPERATURE = 0.0
THINKING_BUDGET = 2048    # LOW: thinking minimum (512 token). 0=off, -1=dinamis(high), mis 8192=fixed besar

# ── Per-route model + thinking (HYBRID): hemat token ──────────────────
# Halaman teks bersih -> model murah, thinking OFF.
# Halaman ribet (math-2D / multi-solusi) -> model kuat + thinking high (cegah
# model bawah halusinasi bikin soal palsu). Hanya backend "gemini" yang pakai ini.
TEXT_MODEL    = "gemini-2.5-flash-lite"   # route 'text' (teks bersih) -> murah
TEXT_THINKING = 0                          # WAJIB 0: matikan thinking flash-lite
HARD_MODEL    = "gemini-3.1-flash-lite"    # route 'image' (halaman ribet)
HARD_THINKING = -1                         # thinking high (dinamis); atau fix mis 8192
SKIP_SCAN     = True                       # halaman tanpa text-layer (scan) -> BUANG, tak kirim VLM
MAX_RETRIES = 6          # retry hormati retryDelay dari API
MAX_WORKERS = 4          # turun dari 8: 8 worker hajar model overloaded -> perparah 503

# ── Resume ────────────────────────────────────────────────────────────
OVERWRITE = False        # True = proses ulang PDF yg outputnya sudah ada

API_KEY = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
assert USE_QWEN_VLM or USE_GEMINI_CLI or API_KEY, \
    "Set GEMINI_API_KEY di .env, atau USE_GEMINI_CLI=True, atau USE_QWEN_VLM=True."

print("ON_KAGGLE    :", ON_KAGGLE)
print("RAW_DIR      :", RAW_DIR, "(ada)" if RAW_DIR.exists() else "(TIDAK ADA)")
print("OUT_DIR      :", OUT_DIR)
print("Backend/model:", BACKEND, "/", MODEL_ID)
print(f"Rate limit   : sleep {SLEEP_BETWEEN}s/request, max {MAX_RETRIES} retry")

ON_KAGGLE    : True
RAW_DIR      : G:\.shortcut-targets-by-id\1Y8RaC_XzbNTbjpMEdlrkNvL3k5GtC-0u\DATA_NLP\raw (ada)
OUT_DIR      : C:\Users\Henry\Documents\KULIAH\sem_4\NLP\FP\data\extracted_vlm
Backend/model: gemini / gemini-3.1-flash-lite
Rate limit   : sleep 0.0s/request, max 6 retry


## 3. Prompt ekstraksi

Satu prompt berbahasa Indonesia yang memberi tahu VLM cara memisahkan soal vs
materi dan membuang noise. Output dipaksa JSON array.

In [4]:
EXTRACTION_PROMPT = """Kamu adalah ekstraktor data dari SATU halaman buku/kumpulan soal matematika berbahasa Indonesia.
Aku beri satu gambar halaman. Keluarkan HANYA sebuah JSON array (tanpa teks lain, tanpa pagar ```).

Setiap elemen array adalah salah satu dari TIGA tipe:

1) SOAL — sesuatu yang harus diselesaikan (soal latihan, CONTOH SOAL, soal olimpiade, pertanyaan).
   WAJIB memuat kalimat pertanyaan/perintah ("Ada berapa...", "Tentukan...", "Hitunglah...", "Buktikan..."):
{
  "type": "soal",
  "nomor": "<nomor soal jika ada, mis. '1', '12a'; jika tidak ada kosongkan ''>",
  "question": "<teks soal LENGKAP kata demi kata; tulis notasi matematika sebagai LaTeX inline $...$>",
  "answer": "<jawaban akhir jika tertera di halaman; jika tidak ada ''>",
  "steps": "<langkah penyelesaian / pembahasan / cara jika ada di halaman; jika tidak ada ''>",
  "has_image": <true jika soal memuat/bergantung pada gambar/diagram/grafik, selain itu false>
}

2) LANJUTAN — sambungan PEMBAHASAN/penyelesaian dari soal di HALAMAN SEBELUMNYA, yang di halaman ini
   TIDAK ADA kalimat soal/pertanyaan barunya (cuma lanjutan hitungan/langkah/jawaban akhir):
{
  "type": "lanjutan",
  "steps": "<lanjutan langkah/pembahasan di halaman ini; LaTeX inline $...$>",
  "answer": "<jawaban akhir jika muncul di sini; jika tidak ada ''>"
}

3) MATERI — teori, definisi, teorema, sifat, rumus, penjelasan murni (BUKAN soal, TIDAK ada pertanyaan):
{
  "type": "materi",
  "judul": "<judul/subbab jika ada; jika tidak ada ''>",
  "content": "<isi materi; notasi matematika sebagai LaTeX inline $...$>"
}

ATURAN PENTING:
- WAJIB transkrip SEMUA teks soal apa adanya. Jangan melewatkan soal yang ada teksnya.
- Kalau ada "Contoh", "Contoh 1/2/3", "Contoh Soal", "Problem", atau pertanyaan yang DISERTAI penyelesaian
  -> itu type "soal" (isi question + steps + answer bila ada). JANGAN dilabel materi.
- LARANGAN KERAS #1: field "question" pada type "soal" WAJIB berisi kalimat pertanyaan/perintah yang utuh.
  DILARANG menaruh fragmen pembahasan (mis. "Maka kita hitung $f(14)$.", "Jadi $f(2)=2$.", "Solusi 2 :")
  sebagai "question". Kalau di halaman ini hanya ada lanjutan hitungan TANPA kalimat soal baru
  -> pakai type "lanjutan" (taruh teks itu di "steps"), JANGAN type "soal".
- PECAHAN & FAKTORIAL: transkrip SANGAT hati-hati. Pecahan tulis $\frac{pembilang}{penyebut}$; pastikan pembilang & penyebut TIDAK tertukar dan konsisten dengan hasil yang tertera (mis. 7!/(7!0!)=1, BUKAN 7!/(1!6!)). Kalau ragu baca angka pecahan/eksponen, tulis apa adanya JANGAN mengarang nilai.
- LARANGAN KERAS #2: field "question" TIDAK BOLEH hanya berisi "[Gambar: ...]". Soal SELALU punya kalimat
  pertanyaan/perintah dalam teks. "[Gambar: ...]" hanya TAMBAHAN di dalam question, bukan pengganti teks soal.
- Jika soal butuh gambar, set has_image=true DAN sisipkan deskripsi singkat diagram dengan format "[Gambar: ...]"
  SETELAH teks soalnya, bukan menggantikan teks soal.
- ABAIKAN noise: header/footer, nomor halaman, watermark, nama pemateri/penulis, URL situs, "Halaman X dari Y".
- Pisahkan tiap soal jadi elemen array tersendiri (ikuti penomoran / penanda "Contoh").
- Pertahankan notasi matematika seakurat mungkin sebagai LaTeX. JANGAN mengarang soal/jawaban yang tidak ada.
- Jika halaman tidak memuat soal maupun materi bermakna (sampul, daftar isi, halaman kosong, gambar dekoratif),
  keluarkan array kosong [].

Keluarkan HANYA JSON array yang valid."""


## 4. Backend VLM (`call_vlm`)

Seam tunggal. Default Gemini. Stub Qwen disertakan supaya swap = ganti `BACKEND`
tanpa menyentuh sisa pipeline.

In [5]:
import os
import sys
import shutil
import tempfile
import subprocess
from pathlib import Path

# ── Backend A: Gemini API key + SDK ───────────────────────────────────
import threading as _threading
_tls = _threading.local()   # client per-thread -> aman utk eksekusi paralel (hindari 'client has been closed')

def _get_gemini_client():
    from google import genai
    c = getattr(_tls, 'gclient', None)
    if c is None:
        c = genai.Client(api_key=API_KEY)
        _tls.gclient = c
    return c

def _thinking_kwargs(genai_types, budget="__use_global__"):
    """thinking_config utk Gemini 3.x; aman kalau SDK/model lama (return {}).""" 
    if budget == "__use_global__":
        budget = globals().get("THINKING_BUDGET")
    if budget is None:
        return {}
    try:
        return {"thinking_config": genai_types.ThinkingConfig(thinking_budget=int(budget))}
    except Exception:
        return {}


def _gemini_call(pil_image, prompt: str, model: str = None, thinking="__use_global__") -> str:
    from google.genai import types as genai_types
    resp = _get_gemini_client().models.generate_content(
        model=model or MODEL_ID,
        contents=[prompt, pil_image],
        config=genai_types.GenerateContentConfig(
            temperature=TEMPERATURE,
            response_mime_type="application/json",
            **_thinking_kwargs(genai_types, thinking),
        ),
    )
    return resp.text or ""

def _gemini_text(prompt: str, model: str = None, thinking="__use_global__") -> str:
    from google.genai import types as genai_types
    resp = _get_gemini_client().models.generate_content(
        model=model or MODEL_ID,
        contents=[prompt],
        config=genai_types.GenerateContentConfig(temperature=TEMPERATURE, **_thinking_kwargs(genai_types, thinking)),
    )
    return resp.text or ""


# ── Backend B: Gemini CLI ─────────────────────────────────────────────
def _resolve_gemini_bin() -> str:
    cand = shutil.which("gemini")
    if cand and cand.lower().endswith(".ps1"):
        alt = cand[:-4] + ".cmd"
        if os.path.exists(alt):
            cand = alt
    return cand or "gemini"

_GEMINI_BIN = _resolve_gemini_bin()

def _gemini_cli_run(args_extra, full_prompt):
    args = [_GEMINI_BIN, "-m", MODEL_ID, "-p", full_prompt]
    if sys.platform == "win32" and _GEMINI_BIN.lower().endswith(".cmd"):
        args = ["cmd", "/c"] + args
    r = subprocess.run(args, capture_output=True, text=True,
                       encoding="utf-8", errors="replace", timeout=240)
    if r.returncode != 0:
        raise RuntimeError(f"gemini CLI exit {r.returncode}: {(r.stderr or '')[:300]}")
    return r.stdout or ""

def _gemini_cli_call(pil_image, prompt: str) -> str:
    with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as f:
        tmp = f.name
    try:
        pil_image.save(tmp, format="PNG")
        full = f"{prompt}\n\nBaca gambar halaman berikut: @{Path(tmp).as_posix()}"
        return _gemini_cli_run(None, full)
    finally:
        try:
            os.unlink(tmp)
        except OSError:
            pass

def _gemini_cli_text(prompt: str) -> str:
    return _gemini_cli_run(None, prompt)


# ── Backend C: Qwen2.5-VL via GPU (lokal / Kaggle) ────────────────────
_qwen_model = None
_qwen_processor = None

def _load_qwen():
    global _qwen_model, _qwen_processor
    if _qwen_model is not None:
        return
    import torch
    from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
    print(f"[qwen] memuat {MODEL_ID} (4-bit)... sekali di awal, agak lama.")
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.float16)
    _qwen_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID, quantization_config=bnb, device_map="auto", torch_dtype=torch.float16)
    _qwen_processor = AutoProcessor.from_pretrained(
        MODEL_ID, min_pixels=256 * 28 * 28, max_pixels=1280 * 28 * 28)
    print("[qwen] siap.")

def _qwen_call(pil_image, prompt: str) -> str:
    _load_qwen()
    import torch
    from qwen_vl_utils import process_vision_info
    messages = [{"role": "user", "content": [
        {"type": "image", "image": pil_image},
        {"type": "text", "text": prompt},
    ]}]
    text = _qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = _qwen_processor(text=[text], images=image_inputs, videos=video_inputs,
                             padding=True, return_tensors="pt").to(_qwen_model.device)
    with torch.no_grad():
        gen = _qwen_model.generate(**inputs, max_new_tokens=2048, do_sample=False)
    out = _qwen_processor.batch_decode(gen[:, inputs.input_ids.shape[1]:],
                                       skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    return out

def _qwen_text(prompt: str, max_new_tokens: int = 8) -> str:
    _load_qwen()
    import torch
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    text = _qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = _qwen_processor(text=[text], return_tensors="pt").to(_qwen_model.device)
    with torch.no_grad():
        gen = _qwen_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return _qwen_processor.batch_decode(gen[:, inputs.input_ids.shape[1]:],
                                        skip_special_tokens=True)[0]


# ── Dispatcher ────────────────────────────────────────────────────────
def call_vlm(pil_image, prompt: str, model: str = None, thinking="__use_global__") -> str:
    """VLM dengan gambar (extraction)."""
    if BACKEND == "gemini":     return _gemini_call(pil_image, prompt, model, thinking)
    if BACKEND == "gemini_cli": return _gemini_cli_call(pil_image, prompt)
    if BACKEND == "qwen":       return _qwen_call(pil_image, prompt)
    raise ValueError(f"BACKEND tidak dikenal: {BACKEND}")

def vlm_text(prompt: str, model: str = None, thinking="__use_global__") -> str:
    """Inferensi TEKS-only sesuai BACKEND aktif (dipakai validity check 11b)."""
    if BACKEND == "gemini":     return _gemini_text(prompt, model, thinking)
    if BACKEND == "gemini_cli": return _gemini_cli_text(prompt)
    if BACKEND == "qwen":       return _qwen_text(prompt)
    raise ValueError(f"BACKEND tidak dikenal: {BACKEND}")


print("Backend aktif:", BACKEND)
if BACKEND == "gemini_cli":
    print("Gemini CLI   :", _GEMINI_BIN)

Backend aktif: gemini


## 5. Render halaman & parsing JSON

`PyMuPDF` me-render tiap halaman jadi gambar (tanpa dependency sistem seperti
poppler). Parser JSON dibuat tahan terhadap pagar ```json dan teks liar.

In [6]:
import io
import json
import re
import random
import time
import fitz  # PyMuPDF
from PIL import Image


def page_to_image(page, dpi: int = DPI) -> Image.Image:
    pix = page.get_pixmap(dpi=dpi)
    return Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")


_FENCE_RE = re.compile(r"```(?:json)?\s*(.*?)\s*```", re.DOTALL)

def parse_json_array(text: str):
    """Robust: kembalikan list[dict]. [] kalau gagal."""
    if not text:
        return []
    t = text.strip()
    m = _FENCE_RE.search(t)
    if m:
        t = m.group(1).strip()
    # ambil dari kurung siku pertama sampai terakhir
    i, j = t.find("["), t.rfind("]")
    if i != -1 and j != -1 and j > i:
        t = t[i : j + 1]
    try:
        data = json.loads(t)
    except json.JSONDecodeError:
        return []
    if isinstance(data, dict):
        data = [data]
    return [d for d in data if isinstance(d, dict)]


# Ambil saran tunggu dari pesan error Gemini: "'retryDelay': '40s'" / "retry in 44.2s".
_RETRY_RE = re.compile(r"retry(?:Delay)?['\":\s]*in?\s*['\"]?(\d+(?:\.\d+)?)\s*s", re.IGNORECASE)

def _suggested_wait(err) -> float:
    m = _RETRY_RE.search(str(err))
    return float(m.group(1)) if m else 0.0


def call_vlm_with_retry(pil_image, prompt: str, model=None, thinking="__use_global__") -> str:
    """Return teks. RAISE RuntimeError kalau gagal semua percobaan, supaya
    process_pdf tidak menulis output bolong yang ke-skip saat resume."""
    last = None
    for attempt in range(MAX_RETRIES):
        try:
            out = call_vlm(pil_image, prompt, model, thinking)
            if SLEEP_BETWEEN:
                time.sleep(SLEEP_BETWEEN)
            return out
        except Exception as e:  # 429 rate limit / 503 server sibuk / transient
            last = e
            backoff = (2 ** attempt) * max(SLEEP_BETWEEN, 1.0)
            # 429 kasih retryDelay ~20-45s; tunggu segitu, bukan cuma backoff kecil.
            wait = max(backoff, _suggested_wait(e) + 2.0)
            wait += random.uniform(0, wait * 0.5)  # jitter: cegah 4 worker retry barengan (thundering herd)
            print(f"    retry {attempt+1}/{MAX_RETRIES}: {str(e)[:110]} (sleep {wait:.0f}s)")
            time.sleep(wait)
    raise RuntimeError(f"VLM gagal setelah {MAX_RETRIES} percobaan: {last}")

## 6. Bangun record & proses per-PDF

Skema soal selaras dengan pipeline yang ada (`question/answer/steps/source_file/
source_page/question_id/has_image`) + field `type` untuk memisahkan soal vs
materi. Tiap PDF → satu file `.jsonl`. Bisa di-resume (skip yang sudah ada).

In [7]:
# ── Bangun record per-PDF: lanjutan-merge + HYBRID routing + PARALEL ─────
# 1) page_items_to_records: soal / lanjutan / materi.
# 2) _merge_lanjutan: gabung pembahasan lintas-halaman ke soal sebelumnya.
# 3) route_page: teks-bersih -> ekstraksi TEKS (murah); math-2D/rumus-gambar/scan -> VLM image.
# 4) process_pdf: request halaman dijalankan PARALEL (MAX_WORKERS) per-PDF -> jauh lebih cepat.

import re
import random
import concurrent.futures as _cf

USE_HYBRID      = True    # False = selalu image (perilaku lama)
IMG_COV_THR     = 0.12    # >12% luas halaman gambar -> image (diagram/rumus-gambar)
SHORT_LINE_THR  = 0.30    # >30% baris <=3 char -> image (pecahan vertikal / multi-kolom pecah)
MIN_TEXT_CHARS  = 40      # teks < 40 char -> scan/kosong -> image
IMG_ROUTE_DPI   = 200     # DPI render image-route (>150 = pecahan/faktorial kecil kebaca)

# ── Hemat token: skip halaman non-soal + drop output materi ───────────
SOAL_ONLY_FILTER    = True              # halaman tanpa marker soal -> TIDAK dikirim ke VLM
SOAL_FOLDERS_ALWAYS = {"osn"}           # folder kumpulan-soal murni: selalu proses penuh
SKIP_MATERI         = True              # model tidak output type "materi" (hemat output token)

# marker soal: perintah/pertanyaan + penanda latihan/contoh + nomor soal. Sengaja permisif.
_SOAL_RE = re.compile(
    r"(latihan|soal|contoh|problem|tentukan|hitung|buktikan|carilah|"
    r"cari|selesaikan|nyatakan|berapa|tunjukkan|misalkan|"
    r"jika.*maka|^\s*\d+\s*[.)]\s)",
    re.IGNORECASE | re.MULTILINE,
)

def page_has_soal(text: str, folder: str, tlen: int) -> bool:
    """True = kandidat soal (kirim VLM). False = teori murni (skip, no API call)."""
    if not SOAL_ONLY_FILTER:
        return True
    if folder in SOAL_FOLDERS_ALWAYS:
        return True
    if tlen < MIN_TEXT_CHARS:           # scan/no-text: tak bisa dinilai -> kirim (aman)
        return True
    return bool(_SOAL_RE.search(text))

# drop materi dari output (idempoten; aman di-rerun)
if SKIP_MATERI and "OVERRIDE-SKIP-MATERI" not in EXTRACTION_PROMPT:
    EXTRACTION_PROMPT = EXTRACTION_PROMPT + chr(10) + chr(10) + (
        "OVERRIDE-SKIP-MATERI: JANGAN keluarkan elemen type materi sama sekali. Halaman yang isinya teori/definisi/teorema/penjelasan TANPA soal -> kembalikan array kosong []. Tetap keluarkan type soal dan lanjutan sesuai aturan di atas."
    )


def _subject_for(pdf_path: Path) -> str:
    """Ambil 'subject' dari source.json di folder yang sama, jika ada."""
    sj = pdf_path.parent / "source.json"
    if sj.exists():
        try:
            return json.loads(sj.read_text(encoding="utf-8")).get("subject", "")
        except Exception:
            return ""
    return ""


def page_items_to_records(items, pdf_path, page_num, subject):
    """type 'lanjutan' dikeluarkan apa adanya; digabung ke soal di _merge_lanjutan."""
    records = []
    n_soal = n_materi = 0
    for it in items:
        typ = (it.get("type") or "").lower()
        if typ == "soal":
            n_soal += 1
            records.append({
                "type": "soal",
                "question": (it.get("question") or "").strip(),
                "answer":   (it.get("answer") or "").strip(),
                "steps":    (it.get("steps") or "").strip(),
                "nomor":    str(it.get("nomor") or "").strip(),
                "has_image": bool(it.get("has_image", False)),
                "source_file":   pdf_path.name,
                "source_folder": pdf_path.parent.name,
                "subject":       subject,
                "source_page":   page_num,
                "question_id":   f"{pdf_path.stem}_p{page_num}_q{n_soal:02d}",
                "extraction_method": "vlm",
            })
        elif typ == "lanjutan":
            records.append({
                "type": "lanjutan",
                "steps":  (it.get("steps") or "").strip(),
                "answer": (it.get("answer") or "").strip(),
                "source_page": page_num,
            })
        elif typ == "materi":
            n_materi += 1
            records.append({
                "type": "materi",
                "judul":   (it.get("judul") or "").strip(),
                "content": (it.get("content") or "").strip(),
                "source_file":   pdf_path.name,
                "source_folder": pdf_path.parent.name,
                "subject":       subject,
                "source_page":   page_num,
                "question_id":   f"{pdf_path.stem}_p{page_num}_m{n_materi:02d}",
                "extraction_method": "vlm",
            })
    return records


def _merge_lanjutan(all_records):
    """Gabung tiap 'lanjutan' ke soal terakhir sebelumnya (lintas-halaman). steps
    di-append, answer diisi kalau soal belum punya. Lanjutan yatim dibuang."""
    out, last_soal = [], None
    n_merged = n_orphan = 0
    for r in all_records:
        if r.get("type") == "lanjutan":
            if last_soal is None:
                n_orphan += 1
                continue
            if r.get("steps"):
                last_soal["steps"] = (last_soal["steps"] + "\n" + r["steps"]).strip()
            if r.get("answer") and not last_soal["answer"]:
                last_soal["answer"] = r["answer"]
            n_merged += 1
        else:
            if r.get("type") == "soal":
                last_soal = r
            out.append(r)
    return out, n_merged, n_orphan


# ── HYBRID routing ────────────────────────────────────────────────────
def page_signals(page):
    text = page.get_text()
    rect = page.rect
    page_area = (rect.width * rect.height) or 1.0
    img_area = 0.0
    for info in page.get_image_info():
        bb = info.get("bbox")
        if bb:
            img_area += max(0.0, bb[2] - bb[0]) * max(0.0, bb[3] - bb[1])
    img_cov = min(img_area / page_area, 1.0)
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    short_ratio = (sum(1 for l in lines if len(l) <= 3) / len(lines)) if lines else 1.0
    return text, img_cov, short_ratio, len(text)


def route_page(page):
    """Return (route, alasan, page_text). route in {'skip','text','image'}.
      skip  = scan / tanpa text-layer  -> dibuang (kalau SKIP_SCAN).
      text  = teks bersih              -> TEXT_MODEL, thinking off (murah).
      image = math-2D/diagram (ribet)  -> HARD_MODEL, thinking high, render gambar."""
    text, img_cov, short_ratio, tlen = page_signals(page)
    if tlen < MIN_TEXT_CHARS:
        return ("skip" if SKIP_SCAN else "image"), f"teks {tlen}c (scan/kosong)", text
    if img_cov > IMG_COV_THR:
        return "image", f"gambar {img_cov:.0%}", text
    if short_ratio > SHORT_LINE_THR:
        return "image", f"baris pendek {short_ratio:.0%} (math-2D?)", text
    return "text", "teks bersih", text


def _text_prompt(page_text: str) -> str:
    """Prompt ekstraksi versi TEKS (reuse aturan EXTRACTION_PROMPT)."""
    head = EXTRACTION_PROMPT.replace(
        "Aku beri satu gambar halaman.",
        "Aku beri TEKS hasil ekstraksi SATU halaman PDF (urutan baca bisa sedikit kacau).",
    )
    return f'{head}\n\nTEKS HALAMAN:\n"""\n{page_text}\n"""'


def _text_extract_with_retry(page_text: str, model=None, thinking="__use_global__") -> str:
    """vlm_text (mode teks) + retry hormati retryDelay. RAISE kalau gagal semua."""
    prompt = _text_prompt(page_text)
    last = None
    for attempt in range(MAX_RETRIES):
        try:
            out = vlm_text(prompt, model, thinking)
            if SLEEP_BETWEEN:
                time.sleep(SLEEP_BETWEEN)
            return out
        except Exception as e:
            last = e
            wait = max((2 ** attempt) * max(SLEEP_BETWEEN, 1.0), _suggested_wait(e) + 2.0)
            wait += random.uniform(0, wait * 0.5)  # jitter
            time.sleep(wait)
    raise RuntimeError(f"text-extract gagal {MAX_RETRIES}x: {last}")


def _extract_page(route, ptext, pil):
    """Satu request VLM (dipanggil paralel di thread). Return teks mentah."""
    if route == "text":
        return _text_extract_with_retry(ptext, TEXT_MODEL, TEXT_THINKING)   # murah, no-think
    return call_vlm_with_retry(pil, EXTRACTION_PROMPT, HARD_MODEL, HARD_THINKING)  # kuat, thinking high


def process_pdf(pdf_path: Path, max_pages: int | None = None, verbose: bool = True):
    pdf_path = Path(pdf_path)
    out_path = OUT_DIR / f"{pdf_path.parent.name}__{pdf_path.stem}.jsonl"
    if out_path.exists() and not OVERWRITE:
        if verbose:
            print(f"[skip] sudah ada: {out_path.name}")
        return out_path

    subject = _subject_for(pdf_path)
    all_records = []
    n_text = n_image = n_skip = 0
    with fitz.open(pdf_path) as doc:
        n_pages = doc.page_count if max_pages is None else min(max_pages, doc.page_count)
        results = [""] * n_pages
        routes  = [("image", "")] * n_pages
        chunk = max(MAX_WORKERS * 2, 1)   # render+jalankan per-chunk (batasi RAM gambar)
        for base in range(0, n_pages, chunk):
            tasks = []
            for pidx in range(base, min(base + chunk, n_pages)):
                page = doc[pidx]
                _ptxt = page.get_text()
                if not page_has_soal(_ptxt, pdf_path.parent.name, len(_ptxt)):
                    routes[pidx] = ("skip", "tak ada marker soal")
                    continue
                if USE_HYBRID:
                    route, why, ptext = route_page(page)
                else:
                    route, why, ptext = "image", "hybrid off", None
                if route == "skip":                       # scan / tanpa text-layer -> buang
                    routes[pidx] = ("skip", why)
                    continue
                pil = page_to_image(page, dpi=IMG_ROUTE_DPI) if route == "image" else None
                routes[pidx] = (route, why)
                tasks.append((pidx, route, ptext, pil))
            with _cf.ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
                futs = {ex.submit(_extract_page, r, pt, pil): pidx
                        for (pidx, r, pt, pil) in tasks}
                for fut in _cf.as_completed(futs):
                    pidx = futs[fut]
                    try:
                        results[pidx] = fut.result()
                    except Exception as e:
                        results[pidx] = ""
                        print(f"  ! hal {pidx+1} gagal: {str(e)[:100]}")

    # assemble urut + merge lanjutan
    for pidx in range(n_pages):
        route, why = routes[pidx]
        items = parse_json_array(results[pidx] or "")
        recs = page_items_to_records(items, pdf_path, pidx + 1, subject)
        all_records.extend(recs)
        if route == "text":
            n_text += 1
        elif route == "skip":
            n_skip += 1
        else:
            n_image += 1
        if verbose:
            ns = sum(r["type"] == "soal" for r in recs)
            nm = sum(r["type"] == "materi" for r in recs)
            nl = sum(r["type"] == "lanjutan" for r in recs)
            extra = f", {nl} lanjutan" if nl else ""
            print(f"  hal {pidx+1:>3}/{n_pages} [{route:5s}|{why}]: {ns} soal, {nm} materi{extra}")

    all_records, n_merged, n_orphan = _merge_lanjutan(all_records)

    with open(out_path, "w", encoding="utf-8") as f:
        for r in all_records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    if verbose:
        ns = sum(r["type"] == "soal" for r in all_records)
        nm = sum(r["type"] == "materi" for r in all_records)
        note = f" (+{n_merged} lanjutan)" if n_merged else ""
        note += f" ({n_orphan} yatim dibuang)" if n_orphan else ""
        print(f"[ok] {pdf_path.parent.name}/{pdf_path.name}: {ns} soal, {nm} materi "
              f"[teks={n_text} img={n_image} skip={n_skip}] -> {out_path.name}{note}")
    return out_path


print(f"Hybrid={USE_HYBRID} | paralel={MAX_WORKERS} worker | img-DPI={IMG_ROUTE_DPI} | lanjutan-merge ON")
print(f"  text -> {TEXT_MODEL} (think={TEXT_THINKING}) | image -> {HARD_MODEL} (think={HARD_THINKING}) | skip-scan={SKIP_SCAN}")


Hybrid=True | paralel=4 worker | img-DPI=200 | lanjutan-merge ON
  text -> gemini-2.5-flash-lite (think=0) | image -> gemini-3.1-flash-lite (think=-1) | skip-scan=True


In [8]:
# ══════════════════════════════════════════════════════════════════════
# REAL-TIME STREAMING (text-only, incremental, resumable)
# - PDF-level: PDF mayoritas scan (tanpa text-layer) -> SKIP seluruhnya.
# - text-only: cuma halaman ber-text-layer yang dikirim (tanpa render gambar).
# - skip soal yang refer gambar (has_image=True).
# - tiap halaman selesai -> langsung APPEND ke <pdf>.raw.jsonl (+ marker _page_done).
#   Stop kapan saja, jalankan lagi -> lanjut dari halaman terakhir (resume).
# - selesai 1 PDF -> merge lanjutan -> tulis <pdf>.jsonl final (atomic), hapus .raw.
# ══════════════════════════════════════════════════════════════════════
import os

TEXT_PDF_MIN_RATIO = 0.5    # >=50% halaman punya text-layer -> dianggap PDF text. < itu -> scan, skip.
SKIP_IMAGE_SOAL    = True   # buang soal yang has_image=True (butuh gambar)


def classify_pdf_text(pdf_path, sample_pages=20):
    """Return (is_text, ratio). ratio = fraksi halaman (sample) yang punya text-layer."""
    with fitz.open(pdf_path) as doc:
        n = doc.page_count
        if n == 0:
            return False, 0.0
        idxs = list(range(n)) if n <= sample_pages else [int(i * n / sample_pages) for i in range(sample_pages)]
        text_pages = sum(1 for i in idxs if len(doc[i].get_text().strip()) >= MIN_TEXT_CHARS)
    ratio = text_pages / len(idxs)
    return ratio >= TEXT_PDF_MIN_RATIO, ratio


def _load_done_pages(raw_path):
    done = set()
    if raw_path.exists():
        for line in raw_path.read_text(encoding="utf-8").splitlines():
            if not line.strip():
                continue
            try:
                o = json.loads(line)
            except json.JSONDecodeError:
                continue
            if o.get("type") == "_page_done":
                done.add(o.get("source_page"))
    return done


def process_pdf_stream(pdf_path, max_pages=None, verbose=True):
    """Text-only + incremental + resume. Return final Path, atau None kalau PDF di-skip (scan)."""
    pdf_path = Path(pdf_path)
    final = OUT_DIR / f"{pdf_path.parent.name}__{pdf_path.stem}.jsonl"
    raw_dir = OUT_DIR / "_progress"
    raw_dir.mkdir(parents=True, exist_ok=True)
    raw   = raw_dir / f"{pdf_path.parent.name}__{pdf_path.stem}.raw.jsonl"

    if final.exists() and not OVERWRITE:
        if verbose:
            print(f"[skip] selesai: {final.name}")
        return final

    is_text, ratio = classify_pdf_text(pdf_path)
    if not is_text:
        if verbose:
            print(f"[skip-scan] {pdf_path.parent.name}/{pdf_path.name}: text-ratio {ratio:.0%} < {TEXT_PDF_MIN_RATIO:.0%}")
        return None

    if OVERWRITE and raw.exists():
        raw.unlink()
    done_pages = _load_done_pages(raw)

    subject = _subject_for(pdf_path)
    folder = pdf_path.parent.name
    n_new = 0

    with fitz.open(pdf_path) as doc:
        n_pages = doc.page_count if max_pages is None else min(max_pages, doc.page_count)
        # kumpulkan halaman yang perlu di-extract (text-layer + marker soal + belum done)
        todo, skip_pages = [], []
        for pidx in range(n_pages):
            page_num = pidx + 1
            if page_num in done_pages:
                continue
            ptxt = doc[pidx].get_text()
            if len(ptxt.strip()) < MIN_TEXT_CHARS:      # halaman scan/kosong -> bukan text -> skip
                skip_pages.append(page_num); continue
            if not page_has_soal(ptxt, folder, len(ptxt)):
                skip_pages.append(page_num); continue
            todo.append((page_num, ptxt))

        f = open(raw, "a", encoding="utf-8")
        try:
            for page_num in skip_pages:                  # tandai halaman skip biar resume gak ulang
                f.write(json.dumps({"type": "_page_done", "source_page": page_num}, ensure_ascii=False) + "\n")
            f.flush()
            with _cf.ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
                futs = {ex.submit(_text_extract_with_retry, pt): pn for pn, pt in todo}
                for fut in _cf.as_completed(futs):
                    pn = futs[fut]
                    try:
                        raw_out = fut.result()
                    except Exception as e:
                        raw_out = ""
                        print(f"  ! hal {pn} gagal: {str(e)[:90]}")
                    items = parse_json_array(raw_out or "")
                    recs = page_items_to_records(items, pdf_path, pn, subject)
                    if SKIP_IMAGE_SOAL:
                        recs = [r for r in recs if not (r.get("type") == "soal" and r.get("has_image"))]
                    for r in recs:
                        f.write(json.dumps(r, ensure_ascii=False) + "\n")
                    f.write(json.dumps({"type": "_page_done", "source_page": pn}, ensure_ascii=False) + "\n")
                    f.flush()                            # <- tiap halaman langsung kesimpen
                    n_new += 1
                    if verbose:
                        ns = sum(1 for r in recs if r["type"] == "soal")
                        print(f"  hal {pn} -> {ns} soal  [{n_new}/{len(todo)}]")
        finally:
            f.close()

    # finalize: baca semua raw, buang marker, sort per-halaman, merge lanjutan, tulis final atomic
    all_records = []
    for line in raw.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        o = json.loads(line)
        if o.get("type") == "_page_done":
            continue
        all_records.append(o)
    all_records.sort(key=lambda r: r.get("source_page", 0))
    all_records, n_merged, n_orphan = _merge_lanjutan(all_records)

    tmp = str(final) + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        for r in all_records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    os.replace(tmp, final)
    if raw.exists():
        raw.unlink()

    if verbose:
        ns = sum(1 for r in all_records if r["type"] == "soal")
        note = f" (+{n_merged} lanjutan)" if n_merged else ""
        print(f"[ok] {folder}/{pdf_path.name}: {ns} soal{note} -> {final.name}")
    return final


print(f"Streaming siap | text-only | skip-scan<{TEXT_PDF_MIN_RATIO:.0%} | skip-image-soal={SKIP_IMAGE_SOAL} | resume ON")


Streaming siap | text-only | skip-scan<50% | skip-image-soal=True | resume ON


## 7. Daftar PDF & perkiraan skala

Lihat berapa PDF & total halaman sebelum batch (untuk perkiraan waktu/biaya).

In [9]:
# Folder yang diproses DULUAN (prioritas). Sisanya menyusul, urut nama.
PRIORITY_FOLDERS = ("osn",)

def list_pdfs(raw_dir: Path):
    def _key(p):
        prio = PRIORITY_FOLDERS.index(p.parent.name) if p.parent.name in PRIORITY_FOLDERS else len(PRIORITY_FOLDERS)
        return (prio, str(p).lower())
    return sorted(raw_dir.rglob("*.pdf"), key=_key)

pdfs = list_pdfs(RAW_DIR)
total_pages = 0
print(f"{len(pdfs)} PDF ditemukan di {RAW_DIR}\n")
by_folder = {}
for p in pdfs:
    by_folder.setdefault(p.parent.name, 0)
    by_folder[p.parent.name] += 1
for folder, n in sorted(by_folder.items()):
    print(f"  {n:>3} PDF  {folder}")

# Hitung total halaman (cepat, hanya buka metadata)
for p in pdfs:
    try:
        with fitz.open(p) as d:
            total_pages += d.page_count
    except Exception as e:
        print(f"  ! gagal buka {p.name}: {e}")
print(f"\nTotal halaman (≈ jumlah request VLM): {total_pages}")


95 PDF ditemukan di G:\.shortcut-targets-by-id\1Y8RaC_XzbNTbjpMEdlrkNvL3k5GtC-0u\DATA_NLP\raw

    1 PDF  aljabar_linear_dasar
    1 PDF  aljabar_linear_uinsgd
    1 PDF  analisa_vektor_uki
    1 PDF  buku_guru_matematika
    1 PDF  gdrive_155ne
    1 PDF  gdrive_1vmka
    1 PDF  kalkulus_1_polsri
    1 PDF  kalkulus_2_bbg
    1 PDF  kalkulus_unipahlawan
    1 PDF  kombinatorika_aljabar_itb
    1 PDF  kombinatorika_itb
    1 PDF  matematika_kelas_xii
    1 PDF  matematika_tingkat_lanjut_xii
    1 PDF  materi_pembelajaran_uki
   80 PDF  osn
    1 PDF  statistika_pendidikan

Total halaman (≈ jumlah request VLM): 6500


## 8. PROTOTYPE — uji di beberapa PDF dulu

Jalankan di sedikit PDF (campur ebook + kumpulan soal), batasi halaman, lalu
**periksa hasilnya** sebelum batch penuh. Sesuaikan `SAMPLE` & `MAX_PAGES`.

In [10]:
# Pilih sampel: 1 ebook + beberapa kumpulan soal osn (sesuaikan nama folder bila perlu)
def pick_samples():
    picks = []
    # satu dari folder ebook/buku ajar mana pun selain osn
    ebook = next((p for p in pdfs if p.parent.name != "osn"), None)
    if ebook:
        picks.append(ebook)
    # dua dari osn
    picks += [p for p in pdfs if p.parent.name == "osn"][:2]
    return picks or pdfs[:3]


def rel_path(p):
    """Path ringkas utk print; tahan beda drive (RAW_DIR di G:, proyek di C:)."""
    for base in (RAW_DIR, PROJECT_ROOT):
        try:
            return p.relative_to(base)
        except ValueError:
            continue
    return p.name

SAMPLE = pick_samples()
MAX_PAGES = 4   # batasi halaman per PDF saat prototype

print("Sampel:")
for p in SAMPLE:
    print("  ", rel_path(p))
print()

_orig_overwrite = OVERWRITE
OVERWRITE = True  # selalu proses ulang saat prototype
for p in SAMPLE:
    process_pdf_stream(p, max_pages=MAX_PAGES, verbose=True)
OVERWRITE = _orig_overwrite

Sampel:
   aljabar_linear_dasar\aljabar-linear-dasar.pdf
   osn\unknown_Download_File_1-MDRMez.pdf
   osn\unknown_Download_File_1-pmlCix.pdf

[skip-scan] aljabar_linear_dasar/aljabar-linear-dasar.pdf: text-ratio 0% < 50%
  hal 4 -> 1 soal  [1/4]
  hal 1 -> 1 soal  [2/4]
  hal 3 -> 0 soal  [3/4]
  hal 2 -> 0 soal  [4/4]
[ok] osn/unknown_Download_File_1-MDRMez.pdf: 2 soal (+3 lanjutan) -> osn__unknown_Download_File_1-MDRMez.jsonl
  hal 4 -> 0 soal  [1/4]
  hal 1 -> 0 soal  [2/4]
  hal 3 -> 0 soal  [3/4]
  hal 2 -> 0 soal  [4/4]
[ok] osn/unknown_Download_File_1-pmlCix.pdf: 0 soal -> osn__unknown_Download_File_1-pmlCix.jsonl


### Periksa hasil prototype

Cetak beberapa record soal & materi untuk dicek kualitasnya.

In [11]:
def preview_jsonl(stem_folder, n=6):
    path = OUT_DIR / f"{stem_folder}.jsonl"
    if not path.exists():
        print(f"== {path.name}: (belum ada / PDF di-skip scan) =="); return
    rows = [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines()]
    print(f"== {path.name}: {len(rows)} record ==\n")
    for r in rows[:n]:
        if r["type"] == "soal":
            print(f"[SOAL hal {r['source_page']} no {r.get('nomor','')}] img={r['has_image']}")
            print("  Q:", r["question"][:300])
            if r["answer"]: print("  A:", r["answer"][:160])
            if r["steps"]:  print("  cara:", r["steps"][:200], "...")
        else:
            print(f"[MATERI hal {r['source_page']}] {r.get('judul','')}")
            print("  ", r["content"][:280])
        print("-" * 70)

for p in SAMPLE:
    preview_jsonl(f"{p.parent.name}__{p.stem}")
    print("\n")


== aljabar_linear_dasar__aljabar-linear-dasar.jsonl: (belum ada / PDF di-skip scan) ==


== osn__unknown_Download_File_1-MDRMez.jsonl: 2 record ==

[SOAL hal 1 no 1] img=False
  Q: Budi akan menaiki 14 buah anak tangga. Ia dapat melewati 1 atau 2 anak tangga sekaligus. Ada berapa banyak cara Budi melewati 14 anak tangga tersebut ?
  A: 10
  cara: Misalkan $f(n)$ adalah banyaknya cara untuk sampai anak tangga ke-$n$. Untuk sampai anak tangga ke-$n$, maka sebelumnya harus sampai anak tangga ke-$(n-1)$ atau anak tangga ke-$(n-2)$. Maka rumus reku ...
----------------------------------------------------------------------
[SOAL hal 4 no ] img=False
  Q: Bisakah dengan cara inklusi-eksklusi ?
  cara: Jawab 3 : Biasanya kita bagi kasus : Kasus 1, 2222222 artinya dua langkah dua langkah sampai ke atas banyaknya cara 1. Kasus 2, 11222222 banyaknya cara $\frac{8!}{2! \cdot 6!} = 28$. Kasus 3, ........ ...
----------------------------------------------------------------------


== osn__unknown_Do

## 9. BATCH PENUH

Setelah hasil prototype oke, proses semua PDF. Resumable: PDF yang outputnya
sudah ada akan di-skip (`OVERWRITE=False`). Aman dihentikan & dilanjutkan.

In [12]:
# from tqdm.auto import tqdm

# errors = []
# n_done = n_scan = 0
# for p in tqdm(pdfs, desc="PDF (text-only stream)"):
#     try:
#         r = process_pdf_stream(p, max_pages=None, verbose=False)
#         if r is None:
#             n_scan += 1          # PDF scan -> di-skip
#         else:
#             n_done += 1
#     except Exception as e:
#         errors.append((p, e))
#         print(f"! error {p.name}: {str(e)[:120]}")

# print(f"\nSelesai. text-PDF diproses={n_done}, scan-PDF di-skip={n_scan}, error={len(errors)}/{len(pdfs)}")
# if errors:
#     print("Gagal:")
#     for p, e in errors:
#         print("  ", p.name, "->", str(e)[:100])


## Batch Mode (anti-503, 50% lebih murah)

Pengganti loop real-time di atas. Kirim semua halaman jadi batch job async,
poll sampai selesai, lalu assemble pakai helper yang sama.

- **Tanpa 503**: batch antri di pool kapasitas terpisah Google, bukan real-time.
- **50% lebih murah** dari harga normal token.
- **Trade-off**: async — job selesai dalam menit s/d beberapa jam (target < 24 jam).

Jalankan **cell helper + cell driver** di bawah ALIH-ALIH cell loop real-time.
Set `THINKING_BUDGET = 512` (atau `-1`) di cell config kalau mau thinking aktif.

In [13]:
# ══════════════════════════════════════════════════════════════════════
# BATCH MODE (FILE-BASED, 1 job) — anti-503 + anti-429 job-count
# Semua request -> 1 file JSONL -> upload -> 1 batch job -> poll -> download.
# (Versi inline-many-jobs lama kena 429 RESOURCE_EXHAUSTED krn kebanyakan job.)
# ══════════════════════════════════════════════════════════════════════
import io as _io, os, json, time, tempfile
from google.genai import types as gtypes
from google.genai import _common as _gcommon
from google.genai import batches as _gbatches

BATCH_MODEL = "gemini-2.5-flash"   # batch: flash penuh, tanpa takut 503/429
POLL_EVERY  = 30                    # detik antar cek status job

_DONE_STATES = {"JOB_STATE_SUCCEEDED", "JOB_STATE_FAILED", "JOB_STATE_CANCELLED",
                "JOB_STATE_EXPIRED", "JOB_STATE_PARTIALLY_SUCCEEDED"}


def _png_bytes(pil):
    buf = _io.BytesIO(); pil.save(buf, format="PNG")
    return buf.getvalue()


def build_batch_requests(pdf_list, max_pages=None):
    """Return (requests, metas, sizes). Routing & skip SAMA dgn pipeline real-time."""
    from tqdm.auto import tqdm as _tqdm
    requests, metas, sizes = [], [], []
    for pdf_path in _tqdm(pdf_list, desc="build req (render gambar)"):
        subject = _subject_for(pdf_path)
        with fitz.open(pdf_path) as doc:
            n_pages = doc.page_count if max_pages is None else min(max_pages, doc.page_count)
            for pidx in range(n_pages):
                page = doc[pidx]
                ptxt = page.get_text()
                page_num = pidx + 1
                if not page_has_soal(ptxt, pdf_path.parent.name, len(ptxt)):
                    metas.append({"pdf": str(pdf_path), "page": page_num,
                                  "subject": subject, "route": "skip", "req_idx": None})
                    continue
                route, why, ptext = route_page(page) if USE_HYBRID else ("image", "off", None)
                if route == "text":
                    parts = [gtypes.Part.from_text(text=_text_prompt(ptext))]
                    cfg = gtypes.GenerateContentConfig(
                        temperature=TEMPERATURE, **_thinking_kwargs(gtypes))
                    sz = len(ptext or "") + 2000
                else:
                    png = _png_bytes(page_to_image(page, dpi=IMG_ROUTE_DPI))
                    parts = [gtypes.Part.from_text(text=EXTRACTION_PROMPT),
                             gtypes.Part.from_bytes(data=png, mime_type="image/png")]
                    cfg = gtypes.GenerateContentConfig(
                        temperature=TEMPERATURE, response_mime_type="application/json",
                        **_thinking_kwargs(gtypes))
                    sz = len(png) + 3000
                metas.append({"pdf": str(pdf_path), "page": page_num,
                              "subject": subject, "route": route, "req_idx": len(requests)})
                requests.append(gtypes.InlinedRequest(
                    model=BATCH_MODEL,
                    contents=[gtypes.Content(role="user", parts=parts)],
                    config=cfg,
                ))
                sizes.append(sz)
    return requests, metas, sizes


def _write_jsonl(requests, path):
    """Serialize tiap InlinedRequest via serializer SDK sendiri (format dijamin benar)."""
    client = _get_gemini_client()
    with open(path, "w", encoding="utf-8") as f:
        for i, r in enumerate(requests):
            d = _gbatches._InlinedRequest_to_mldev(client._api_client, r)
            d = _gcommon.convert_to_dict(d)               # pydantic (ThinkingConfig dll) -> dict
            d = _gcommon.encode_unserializable_types(d)   # bytes gambar -> base64
            f.write(json.dumps({"key": f"r{i}", "request": d["request"]},
                               ensure_ascii=False) + "\n")


def _resp_text_from_dict(resp):
    """Ambil teks dari dict response hasil batch file."""
    try:
        for cand in resp.get("candidates", []):
            parts = cand.get("content", {}).get("parts", [])
            txt = "".join(p.get("text", "") for p in parts if "text" in p)
            if txt:
                return txt
    except Exception:
        pass
    return ""


def run_batch_file(requests, sizes=None):
    """1 file JSONL -> upload -> 1 job -> poll -> download. Return list[str] (teks) aligned ke requests."""
    client = _get_gemini_client()
    results = [""] * len(requests)
    path = os.path.join(tempfile.gettempdir(), f"batch_in_{int(time.time())}.jsonl")
    print(f"tulis {len(requests)} request -> {path}", flush=True)
    _write_jsonl(requests, path)
    mb = os.path.getsize(path) / 1024 / 1024
    print(f"  JSONL {mb:.1f}MB. upload (sekali)...", flush=True)
    up = client.files.upload(file=path, config=gtypes.UploadFileConfig(mime_type="application/jsonl"))
    print(f"  uploaded {up.name}. buat batch job...", flush=True)
    job = client.batches.create(model=BATCH_MODEL, src=up.name)
    print(f"  job {job.name} dibuat. Poll tiap {POLL_EVERY}s (jangan tutup kernel)...", flush=True)
    t0 = time.time(); i = 0
    while True:
        time.sleep(POLL_EVERY); i += 1
        job = client.batches.get(name=job.name)
        st = job.state.name
        print(f"  poll#{i} t+{int(time.time()-t0)}s: {st}", flush=True)
        if st in _DONE_STATES:
            break
    if job.state.name != "JOB_STATE_SUCCEEDED":
        print(f"  [WARN] job berakhir {job.state.name}: {getattr(job, 'error', None)}", flush=True)
    dest = job.dest
    if dest and getattr(dest, "file_name", None):
        raw = client.files.download(file=dest.file_name).decode("utf-8")
        n = 0
        for line in raw.splitlines():
            if not line.strip():
                continue
            obj = json.loads(line)
            try:
                idx = int(obj.get("key", "")[1:])
            except (ValueError, TypeError):
                continue
            if "response" in obj:
                results[idx] = _resp_text_from_dict(obj["response"]); n += 1
            else:
                results[idx] = ""
        print(f"  download hasil: {n} response ter-parse.", flush=True)
    elif dest and getattr(dest, "inlined_responses", None):
        for k, r in enumerate(dest.inlined_responses):
            ok = r is not None and not getattr(r, "error", None) and r.response is not None
            results[k] = (r.response.text or "") if ok else ""
    print(f"Batch job selesai dalam {int(time.time()-t0)}s.", flush=True)
    return results


def assemble_from_batch(metas, results):
    """Tulis per-PDF jsonl. results = list[str] teks (aligned ke req_idx)."""
    by_pdf = {}
    for m in metas:
        by_pdf.setdefault(m["pdf"], []).append(m)
    for pdf_str, ms in by_pdf.items():
        pdf_path = Path(pdf_str)
        out_path = OUT_DIR / f"{pdf_path.parent.name}__{pdf_path.stem}.jsonl"
        all_records = []
        for m in sorted(ms, key=lambda x: x["page"]):
            if m["req_idx"] is None:
                continue
            items = parse_json_array(results[m["req_idx"]] or "")
            all_records.extend(page_items_to_records(items, pdf_path, m["page"], m["subject"]))
        all_records, n_merged, n_orphan = _merge_lanjutan(all_records)
        with open(out_path, "w", encoding="utf-8") as f:
            for rec in all_records:
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        ns = sum(1 for r in all_records if r["type"] == "soal")
        print(f"[ok-batch] {out_path.name}: {ns} soal (+{n_merged} lanjutan)")


print("Batch helpers (file-based) siap. model:", BATCH_MODEL)


Batch helpers (file-based) siap. model: gemini-2.5-flash


## 9b. md_good -> soal JSONL (mode TEKS, murah)

Untuk md OSN yang lolos `filter_md.py` (bersih): ekstrak soal lewat **mode teks**
(`TEXT_MODEL`, thinking off) — tanpa render gambar, jauh lebih murah dari VLM.
md **hancur** (`md_hancur.txt`) tidak diproses di sini; PDF-nya dikirim ke VLM.

Skema output identik dgn VLM (`source_folder=osn`, `source_file=<stem>.pdf`) supaya
`to_csv` / `dedup` hilir memperlakukannya sama. Tiap chunk dihitung sebagai 1 'halaman'.


In [14]:
# ══ md_good -> soal jsonl (reuse vlm_text/_text_prompt/parse/records) ══
MD_DIR      = Path(r"G:/.shortcut-targets-by-id/1Y8RaC_XzbNTbjpMEdlrkNvL3k5GtC-0u/DATA_NLP/raw/0md/osn_md")
MD_GOOD     = OUT_DIR.parent / "filtered" / "md_good.txt"   # OUT_DIR absolut -> root reliable
MD_FOLDER   = "osn"            # source_folder utk schema (konsisten dgn VLM)
MD_SUBJECT  = "matematika"
CHUNK_CHARS = 6000             # target ukuran chunk teks per request


def _chunk_md(text: str, target: int = CHUNK_CHARS):
    """Split per form-feed kalau ada; else gabung paragraf sampai ~target char.
    Chunk kekecilan digabung supaya tak boros request."""
    if "\f" in text:
        parts = [p for p in text.split("\f") if p.strip()]
    else:
        parts, buf, size = [], [], 0
        for para in text.split("\n\n"):
            buf.append(para); size += len(para) + 2
            if size >= target:
                parts.append("\n\n".join(buf)); buf, size = [], 0
        if buf:
            parts.append("\n\n".join(buf))
    merged, cur = [], ""
    for pt in parts:
        if cur and len(cur) + len(pt) < target:
            cur = (cur + "\n\n" + pt).strip()
        else:
            if cur:
                merged.append(cur)
            cur = pt
    if cur:
        merged.append(cur)
    return merged


def process_md(stem: str, verbose: bool = True):
    md_path  = MD_DIR / f"{stem}.md"
    fake_pdf = RAW_DIR / MD_FOLDER / f"{stem}.pdf"     # Path semu -> schema sama dgn VLM
    out_path = OUT_DIR / f"{MD_FOLDER}__{stem}.jsonl"
    if out_path.exists() and not OVERWRITE:
        if verbose:
            print(f"[skip] sudah ada: {out_path.name}")
        return out_path
    if not md_path.exists():
        print(f"[!] md tak ada: {md_path}"); return None

    chunks = _chunk_md(md_path.read_text(encoding="utf-8", errors="replace"))
    all_records = []
    for ci, ch in enumerate(chunks, 1):
        try:
            raw = _text_extract_with_retry(ch, TEXT_MODEL, TEXT_THINKING)
        except Exception as e:
            print(f"  ! {stem} chunk {ci} gagal: {str(e)[:90]}"); continue
        recs = page_items_to_records(parse_json_array(raw), fake_pdf, ci, MD_SUBJECT)
        all_records += recs
        if verbose:
            ns = sum(r["type"] == "soal" for r in recs)
            print(f"  chunk {ci:>3}/{len(chunks)} [{len(ch)}c]: {ns} soal")

    all_records, n_merged, n_orphan = _merge_lanjutan(all_records)
    with open(out_path, "w", encoding="utf-8") as f:
        for r in all_records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    ns = sum(r["type"] == "soal" for r in all_records)
    if verbose:
        note = f" (+{n_merged} lanjutan)" if n_merged else ""
        print(f"[ok-md] {stem}: {len(chunks)} chunk -> {ns} soal -> {out_path.name}{note}")
    return out_path


def run_md_good(limit=None):
    stems = [s.strip() for s in MD_GOOD.read_text(encoding="utf-8").splitlines() if s.strip()]
    if limit:
        stems = stems[:limit]
    print(f"{len(stems)} md good -> ekstraksi teks ({TEXT_MODEL}, think={TEXT_THINKING})\n")
    for s in stems:
        process_md(s)


print("md->soal siap. Coba 1 dulu: process_md(open(MD_GOOD).read().split()[0])")
print("Lalu semua: run_md_good()   |   batasi: run_md_good(limit=3)")


md->soal siap. Coba 1 dulu: process_md(open(MD_GOOD).read().split()[0])
Lalu semua: run_md_good()   |   batasi: run_md_good(limit=3)


In [15]:
# # ══ DRIVER: proses HANYA md good (mode teks murah) ══════════════════
# # Sumber: data/filtered/md_good.txt (hasil filter_md.py). md hancur TIDAK disini.
# from tqdm.auto import tqdm as _tqdm

# good = [s.strip() for s in MD_GOOD.read_text(encoding="utf-8").splitlines() if s.strip()]
# have_md = [s for s in good if (MD_DIR / f"{s}.md").exists()]
# todo    = [s for s in have_md
#            if OVERWRITE or not (OUT_DIR / f"{MD_FOLDER}__{s}.jsonl").exists()]

# print(f"md good total : {len(good)}")
# print(f"  md ada      : {len(have_md)}  (hilang: {len(good)-len(have_md)})")
# print(f"  belum proses: {len(todo)}  (sudah ada -> skip)\n")

# n_soal_total = 0
# for stem in _tqdm(todo, desc="md good -> soal"):
#     out = process_md(stem, verbose=False)
#     if out and out.exists():
#         n_soal_total += sum(1 for _ in open(out, encoding="utf-8"))
# print(f"\nSelesai. {len(todo)} md diproses (TEXT_MODEL={TEXT_MODEL}, think={TEXT_THINKING}).")
# print(f"Total record tertulis: {n_soal_total}  -> {OUT_DIR}")


In [16]:
# ── DRIVER BATCH (file-based): build -> upload 1 file -> 1 job -> poll -> assemble
# CATATAN: async. 1 job antri di Google (target < 24 jam). Hemat 50% + tanpa 503/429.
# WAJIB uji kecil dulu: set _batch_pdfs = SAMPLE & _batch_maxpages = 4.
_batch_pdfs     = pdfs       # GANTI ke `pdfs` utk full run (HATI2: bisa ribuan request)
_batch_maxpages = None       # GANTI ke None utk semua halaman
CHUNK_SIZE      = 3          # GANTI: Jumlah PDF per job batch agar tidak kena limit 429

# Filter PDF yang sudah selesai agar tidak diproses ulang
if not OVERWRITE:
    _todo_pdfs = []
    for p in _batch_pdfs:
        out_path = OUT_DIR / f"{p.parent.name}__{p.stem}.jsonl"
        if not out_path.exists():
            _todo_pdfs.append(p)
    _batch_pdfs = _todo_pdfs
    print(f"[Info] Sisa {len(_batch_pdfs)} PDF yang belum diproses (yang sudah selesai di-skip).\n")

if not _batch_pdfs:
    print("Semua PDF di list sudah selesai diproses! Lanjut cell 11a.")
else:
    for i in range(0, len(_batch_pdfs), CHUNK_SIZE):
        chunk_pdfs = _batch_pdfs[i:i + CHUNK_SIZE]
        chunk_idx = i // CHUNK_SIZE + 1
        total_chunks = (len(_batch_pdfs) + CHUNK_SIZE - 1) // CHUNK_SIZE
        print(f"\n=== Mengerjakan Batch Chunk {chunk_idx} / {total_chunks} ({len(chunk_pdfs)} PDF) ===")
        
        reqs, metas, sizes = build_batch_requests(chunk_pdfs, max_pages=_batch_maxpages)
        
        n_skip = sum(1 for m in metas if m["route"] == "skip")
        n_img  = sum(1 for m in metas if m["route"] == "image")
        n_txt  = sum(1 for m in metas if m["route"] == "text")
        print(f"{len(reqs)} request  (image={n_img} text={n_txt} skip={n_skip})  "
              f"~{sum(sizes) / 1024 / 1024:.1f}MB payload (sblm base64)")
        
        if not reqs:
            print("Tidak ada request API di chunk ini. Melanjutkan rakit lokal...")
            assemble_from_batch(metas, [])
            continue
        
        try:
            results = run_batch_file(reqs)
            assemble_from_batch(metas, results)
            print(f"--- Chunk {chunk_idx} Selesai ---")
        except Exception as e:
            print(f"\n[ERROR] Chunk gagal karena limit API atau jaringan: {e}")
            print("Silakan tunggu sekitar 15 menit, lalu jalankan ulang cell ini.")
            print("PDF yang SUDAH selesai di chunk sebelumnya AMAN dan otomatis di-skip.")
            break
    
    print("\nSelesai batch. Lanjut cell 11a (rule filter) seperti biasa.")


[Info] Sisa 25 PDF yang belum diproses (yang sudah selesai di-skip).


=== Mengerjakan Batch Chunk 1 / 9 (3 PDF) ===


c:\Users\Henry\Documents\KULIAH\sem_4\NLP\FP\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
build req (render gambar): 100%|██████████| 3/3 [00:54<00:00, 18.29s/it]

341 request  (image=174 text=167 skip=0)  ~49.6MB payload (sblm base64)
tulis 341 request -> C:\Users\Henry\AppData\Local\Temp\batch_in_1780931262.jsonl


  JSONL 66.3MB. upload (sekali)...
  uploaded files/kcllaizyu2l0. buat batch job...
  job batches/bdg16zqr0adob7f9k9nczfq105zbmr7dfg67 dibuat. Poll tiap 30s (jangan tutup kernel)...
  poll#1 t+30s: JOB_STATE_PENDING
  poll#2 t+61s: JOB_STATE_PENDING
  poll#3 t+91s: JOB_STATE_PENDING
  poll#4 t+122s: JOB_STATE_PENDING
  poll#5 t+152s: JOB_STATE_PENDING
  poll#6 t+183s: JOB_STATE_PENDING
  poll#7 t+213s: JOB_STATE_PENDING
  poll#8 t+244s: JOB_STATE_PENDING
  poll#9 t+275s: JOB_STATE_PENDING
  poll#10 t+305s: JOB_STATE_PENDING
  poll#11 t+336s: JOB_STATE_PENDING
  poll#12 t+366s: JOB_STATE_PENDING
  poll#13 t+397s: JOB_STATE_PENDING
  poll#14 t+427s: JOB_STATE_PENDING
  poll#15 t+458s: JOB_STATE_PENDING
  poll#16 t+489s: JOB_STATE_PENDING
  poll#17 t+519s: JOB_STATE_PENDING
  poll#18 t+550s: JOB_STATE_PENDING
  poll#19 t+581s: JOB_STATE_PENDING
  poll#20 t+611s: JOB_STATE_PENDING
  poll#21 t+642s: JOB_STATE_PENDING
  poll#22 t+672s: JOB_STATE_PENDING
  poll#23 t+703s: JOB_STATE_PENDING
  

## 11. Filter pipeline → clean dataset `{soal, cara, jawaban}`

Ambil `_all_soal.jsonl` (soal saja, materi sudah terpisah) lalu saring jadi dataset
bersih siap latih. Empat tahap:

- **11a Rule-based** — buang pilihan ganda, true/false, non-Indonesia. `has_image`
  TIDAK dibuang di sini (dinilai di validity).
- **11b Validity** — pakai ulang model VLM yang sudah dimuat (mode teks) untuk menilai
  apakah soal bisa diselesaikan dari teks saja, termasuk kecukupan deskripsi `[Gambar: ...]`.
- **11c Dedup** — embedding (multilingual-MiniLM) + FAISS, buang duplikat (cosine > 0.92).
  Decontamination mati (belum ada benchmark di `data/eval`).
- **11d Reformat** — tulis `{soal, cara, jawaban}` ke `clean_vlm.jsonl`.

`cara`/`jawaban` yang kosong dibiarkan apa adanya — diisi pada fase CoT synthesis berikutnya.

In [21]:
# ── 11a. Rule-based filter ────────────────────────────────────────────
import re
try:
    from langdetect import detect, LangDetectException
except ImportError:
    detect = None
    LangDetectException = Exception

_MC_RE = re.compile(r"^\s*[A-Ea-e][.)]\s+\S", re.MULTILINE)
_TF_RE = re.compile(
    r"\b(benar\s+atau\s+salah|true\s+or\s+false|"
    r"pernyataan\s+berikut\s+(yang\s+)?(benar|salah)|"
    r"tentukan\s+(mana\s+yang\s+)?(benar|salah))\b",
    re.IGNORECASE,
)

def _is_mc(t):  return len(_MC_RE.findall(t)) >= 3
def _is_tf(t):  return bool(_TF_RE.search(t))
def _is_indo(t):
    if detect is None or len(t.strip()) < 20:
        return True
    try:
        return detect(t) == "id"
    except LangDetectException:
        return True

def passes_rules(rec):
    q = (rec.get("question") or "").strip()
    if len(q) < 10:  return False, "too_short"
    if _is_mc(q):    return False, "multiple_choice"
    if _is_tf(q):    return False, "true_false"
    return True, "ok"

# Muat soal hasil ekstraksi (materi sudah terpisah di _all_materi.jsonl)
soal_path = OUT_DIR / "_all_soal.jsonl"
soal_all = [json.loads(l) for l in soal_path.read_text(encoding="utf-8").splitlines() if l.strip()]

after_rules, rule_rej = [], {}
for r in soal_all:
    ok, why = passes_rules(r)
    if ok:
        after_rules.append(r)
    else:
        rule_rej[why] = rule_rej.get(why, 0) + 1

n_img = sum(1 for r in after_rules if r.get("has_image"))
print(f"11a rule-based: {len(after_rules)}/{len(soal_all)} lolos")
print("   ditolak    :", rule_rej)
print(f"   has_image  : {n_img} (dibiarkan -> dinilai di validity 11b)")

11a rule-based: 3156/3206 lolos
   ditolak    : {'multiple_choice': 32, 'too_short': 17, 'true_false': 1}
   has_image  : 359 (dibiarkan -> dinilai di validity 11b)


In [25]:
# ── 11b. Validity check (heuristik rule-based, TANPA API) ─────
# API Gemini habis -> ganti LLM judge dengan heuristik teks murni (CPU, gratis).
# Menilai apakah soal solvable HANYA dari teks: tolak soal yang merujuk objek
# visual (gambar/grafik/tabel) yang TIDAK terdeskripsi cukup oleh extractor.
import re

# Frasa yang merujuk objek visual eksternal -> butuh lihat gambar asli.
_VIS_REF = re.compile(
    r"(perhatikan\s+(gambar|grafik|tabel|diagram|kurva|bagan|denah|peta))"
    r"|(seperti\s+(tampak\s+)?(pada\s+)?(gambar|grafik|tabel|diagram))"
    r"|((gambar|grafik|tabel|diagram|kurva|bagan|denah|peta|ilustrasi|pola|jaring-jaring)"
    r"\s+(di\s+)?(atas|bawah|samping|berikut|disamping)(\s+ini)?)"
    r"|(berdasarkan\s+(gambar|grafik|tabel|diagram))",
    re.IGNORECASE,
)

# Marker deskripsi gambar dari extractor: "[Gambar: ...]"
_IMG_DESC = re.compile(r"\[Gambar:\s*(.*?)\]", re.IGNORECASE | re.DOTALL)
_MIN_DESC_LEN = 25   # deskripsi [Gambar:] dianggap CUKUP kalau panjang >= sekian char

def _has_sufficient_img_desc(q: str) -> bool:
    descs = _IMG_DESC.findall(q)
    return any(len(d.strip()) >= _MIN_DESC_LEN for d in descs)

def is_valid_soal(rec) -> bool:
    q = (rec.get("question") or "").strip()
    if len(q) < 15:
        return False
    # Buang isi [Gambar: ...] dulu supaya deskripsi itu sendiri tidak ke-flag _VIS_REF.
    q_wo_desc = _IMG_DESC.sub(" ", q)
    needs_visual = bool(_VIS_REF.search(q_wo_desc))
    if needs_visual:
        # Lolos HANYA jika ada deskripsi [Gambar:] cukup panjang utk gantikan gambar.
        return _has_sufficient_img_desc(q)
    return True

from tqdm.auto import tqdm
after_validity = [r for r in tqdm(after_rules, desc="11b validity (heuristik)") if is_valid_soal(r)]

n_drop = len(after_rules) - len(after_validity)
print(f"11b validity (heuristik, tanpa API): {len(after_validity)}/{len(after_rules)} valid"
      f"  ({n_drop} dibuang krn butuh gambar/visual tak terdeskripsi)")


11b validity (heuristik): 100%|██████████| 3156/3156 [00:00<00:00, 21349.91it/s]

11b validity (heuristik, tanpa API): 3110/3156 valid  (46 dibuang krn butuh gambar/visual tak terdeskripsi)


In [26]:
# ── 11c. Dedup (embedding multilingual-MiniLM + FAISS) ────────────────
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

_DEDUP_THRESHOLD = 0.92
_embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

def dedup_keep(texts, threshold=_DEDUP_THRESHOLD):
    """Return indeks yang DISIMPAN (kemunculan pertama menang)."""
    emb = _embedder.encode(
        texts, batch_size=128, normalize_embeddings=True, show_progress_bar=True
    ).astype("float32")
    index = faiss.IndexFlatIP(emb.shape[1])
    keep = []
    for i in range(len(emb)):
        if index.ntotal > 0:
            D, _ = index.search(emb[i:i+1], 1)
            if D[0][0] >= threshold:
                continue
        index.add(emb[i:i+1])
        keep.append(i)
    return keep

if after_validity:
    _keep = dedup_keep([(r.get("question") or "") for r in after_validity])
    after_dedup = [after_validity[i] for i in _keep]
else:
    after_dedup = []
print(f"11c dedup: {len(after_dedup)}/{len(after_validity)} unik "
      f"(buang {len(after_validity) - len(after_dedup)} duplikat)")

Batches: 100%|██████████| 25/25 [00:47<00:00,  1.91s/it]


11c dedup: 2558/3110 unik (buang 552 duplikat)


In [27]:
# ── 11d. Reformat -> {soal, cara, jawaban} + simpan ───────────────────
def to_clean_row(r):
    return {
        "soal":    (r.get("question") or "").strip(),
        "cara":    (r.get("steps")    or "").strip(),
        "jawaban": (r.get("answer")   or "").strip(),
    }

clean_rows = [to_clean_row(r) for r in after_dedup]

clean_path = OUT_DIR / "clean_vlm.jsonl"
with open(clean_path, "w", encoding="utf-8") as f:
    for row in clean_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

n_ans  = sum(1 for r in clean_rows if r["jawaban"])
n_cara = sum(1 for r in clean_rows if r["cara"])
print(f"== clean dataset: {len(clean_rows)} soal -> {clean_path} ==")
print(f"   {len(soal_all)} ekstrak -> {len(after_rules)} rule -> "
      f"{len(after_validity)} valid -> {len(clean_rows)} unik")
print(f"   ada jawaban: {n_ans}   ada cara: {n_cara}   (kosong -> diisi fase CoT)")
print("\nContoh:")
for r in clean_rows[:3]:
    print("  Q:", r["soal"][:140])
    print("     jawaban:", r["jawaban"][:40] or "(kosong)")
    print("     " + "-" * 60)

== clean dataset: 2558 soal -> C:\Users\Henry\Documents\KULIAH\sem_4\NLP\FP\data\extracted_vlm\clean_vlm.jsonl ==
   3206 ekstrak -> 3156 rule -> 3110 valid -> 2558 unik
   ada jawaban: 942   ada cara: 1335   (kosong -> diisi fase CoT)

Contoh:
  Q: Ditentukan titik-titik $O(0, 0)$, $P(-3, 4)$, dan $Q(6, 5)$, carilah komponen dan besarnya vektor berikut ini.
a. $\vec{OP}$
b. $\vec{PQ}$
     jawaban: a. $\vec{OP} = [-3, 4]$, $|\vec{OP}| = 5
     ------------------------------------------------------------
  Q: Contoh 1.4. Operasi vektor. Misalkan $\vec{u} = [-1, 2]$ dan $\vec{v} = [2, 3]$. a. Hitunglah $\vec{u} + \vec{v}$.
     jawaban: $\sqrt{26}$
     ------------------------------------------------------------
  Q: Sederhanakan $2\vec{u} - 3\vec{v}$.
     jawaban: $[-8, -5]$
     ------------------------------------------------------------


## 10. Gabung, pisah soal/materi, ringkasan

Gabungkan semua `.jsonl`, lalu tulis file terpisah untuk `soal` dan `materi`
plus CSV gabungan untuk inspeksi cepat.

In [19]:
import csv

all_rows = []
for jf in sorted(OUT_DIR.glob("*.jsonl")):
    if jf.name.startswith("_") or jf.name.endswith(".raw.jsonl"):
        continue   # skip agregat (_all_*) & file raw resume
    for line in jf.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        o = json.loads(line)
        if o.get("type") == "_page_done":
            continue   # marker progress, bukan data
        all_rows.append(o)

soal   = [r for r in all_rows if r.get("type") == "soal"]
materi = [r for r in all_rows if r.get("type") == "materi"]

def write_jsonl(rows, name):
    path = OUT_DIR / name
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    return path

write_jsonl(soal,   "_all_soal.jsonl")
write_jsonl(materi, "_all_materi.jsonl")

# CSV gabungan (kolom union)
cols = ["type","subject","source_folder","source_file","source_page","question_id",
        "nomor","has_image","question","answer","steps","judul","content"]
csv_path = OUT_DIR / "_all.csv"
with open(csv_path, "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=cols, extrasaction="ignore")
    w.writeheader()
    for r in all_rows:
        w.writerow(r)

print(f"Total record : {len(all_rows)}")
print(f"  soal       : {len(soal)}")
print(f"  materi     : {len(materi)}")
print(f"  soal+jawaban: {sum(1 for r in soal if r['answer'])}")
print(f"  soal+cara   : {sum(1 for r in soal if r['steps'])}")
print(f"  soal bergambar: {sum(1 for r in soal if r['has_image'])}")
print(f"\nDitulis:\n  {OUT_DIR/'_all_soal.jsonl'}\n  {OUT_DIR/'_all_materi.jsonl'}\n  {csv_path}")


Total record : 3211
  soal       : 3206
  materi     : 5
  soal+jawaban: 1169
  soal+cara   : 1670
  soal bergambar: 359

Ditulis:
  C:\Users\Henry\Documents\KULIAH\sem_4\NLP\FP\data\extracted_vlm\_all_soal.jsonl
  C:\Users\Henry\Documents\KULIAH\sem_4\NLP\FP\data\extracted_vlm\_all_materi.jsonl
  C:\Users\Henry\Documents\KULIAH\sem_4\NLP\FP\data\extracted_vlm\_all.csv


## Langkah berikutnya

- `_all_soal.jsonl` siap masuk pipeline filter yang ada
  (`filter_rules` → `filter_validity` → `dedup`). Field sudah selaras.
- Mau pindah backend ke **Qwen2.5-VL** (lokal 4060 / Kaggle)? Isi `_qwen_call`
  lalu set `BACKEND="qwen"`. Sisa notebook tidak berubah.
- Mau **crop gambar diagram** ke file (bukan cuma deskripsi)? Bisa ditambah di
  `process_pdf` pakai `page.get_images()` / bbox PyMuPDF.
